In [ ]:
from aradi import *
import numpy as np

In [ ]:
def aradi_linear_layer(iv, rnd):
    shifts = [[11,8,14],[10,9,11],[9,4,14],[8,9,7]]
    a, b, c = shifts[rnd % 4]
    iv_u = iv[0:16]
    iv_l = iv[16:32]

    ov_u = []
    ov_l = []
    for j in range(16):
        ov_u.append(iv_u[(j+a) % 16] ^ iv_u[j] ^ iv_l[(j+c) % 16])
        ov_l.append(iv_l[(j+a) % 16] ^ iv_l[j] ^ iv_u[(j+b) % 16])

    return ov_u + ov_l

def transpose(M):
    return [list(row) for row in zip(*M)]

def M_T_for_linear_layer(offset):
    output_matrix = []

    for bit in range(31,-1,-1):
        x = 1 << bit
        state = [(x >> i) & 1 for i in range(31, -1, -1)]
        after_linear = aradi_linear_layer(state, offset)
        output_matrix.append(after_linear)

    M_T = transpose(output_matrix)
    return M_T

In [ ]:
output_rounds = 3
output_offset = 1

nb_keybits = [0]*(output_rounds*32)
for i in range(32):
    differences_flat = [[0 for _ in range(32)]]+ [[0 for _ in range(32)] for _ in range(output_rounds)]
    differences_flat[0][i]=1

    shifts = [[11,8,14],[10,9,11],[9,4,14],[8,9,7]]

    differences_pattern = differences_flat.copy()

    for j in range(0, output_rounds):
        a = shifts[(j+output_offset)%4][0]
        b = shifts[(j+output_offset)%4][1]
        c = shifts[(j+output_offset)%4][2]
        iv_u = differences_pattern[j][0:16]
        iv_l = differences_pattern[j][16:32]
        ov_u = differences_pattern[j+1][0:16]
        ov_l = differences_pattern[j+1][16:32]

        active_bits = [(0, k) for k, bit in enumerate(iv_u) if bit] + \
                    [(1, k) for k, bit in enumerate(iv_l) if bit]

        ov_u_final = [0]*16
        ov_l_final = [0]*16

        for layer, bit_idx in active_bits:
            temp_iv_u = [0]*16
            temp_iv_l = [0]*16
            if layer == 0:
                temp_iv_u[bit_idx] = 1
            else:
                temp_iv_l[bit_idx] = 1
            temp_ov_u = [0]*16
            temp_ov_l = [0]*16
            for k in range(16):
                xor_u = temp_iv_u[(k+a)%16] ^ temp_iv_u[k] ^ temp_iv_l[(k+c)%16]
                temp_ov_u[k] = xor_u
                xor_l = temp_iv_l[(k+a)%16] ^ temp_iv_l[k] ^ temp_iv_u[(k+b)%16]
                temp_ov_l[k] = xor_l

            ov_u_final = [ov_u_final[k] | temp_ov_u[k] for k in range(16)]
            ov_l_final = [ov_l_final[k] | temp_ov_l[k] for k in range(16)]

        differences_pattern[j+1][0:16] = ov_u_final
        differences_pattern[j+1][16:32] = ov_l_final

    M_T = M_T_for_linear_layer(output_offset+1)
    mask_1 = np.zeros(32, dtype=int)
    for i in range(32):
        if differences_pattern[1][i]==1:
            mask_1|=M_T[i]

    mask_k2 = mask_1|differences_pattern[2]
    mask_k3 = np.zeros(32, dtype=int)
    M_T = M_T_for_linear_layer(output_offset+2)
    for i in range(32):
        if mask_k2[i]==1:
            mask_k3|=M_T[i]

    mask_k3|=differences_pattern[3]

    nb_keybits[sum(differences_pattern[1])+np.sum(mask_k2)+np.sum(mask_k3)]+=1

print("#key bits | #times")
for k in range(output_rounds*32):
    if nb_keybits[k]!=0:
        print("   ",k,"     ",nb_keybits[k])

#key bits | #times
    39       16
    43       16


In [ ]:
output_offset = 1

nb_keybits = [[0]*(1*32),[0]*(2*32),[0]*(3*32)]
for i in range(32):
    differences_flat = [[0 for _ in range(32)]]+ [[0 for _ in range(32)] for _ in range(output_rounds)]
    differences_flat[0][i]=1

    shifts = [[11,8,14],[10,9,11],[9,4,14],[8,9,7]]

    differences_pattern = differences_flat.copy()

    for j in range(0, output_rounds):
        a = shifts[(j+output_offset)%4][0]
        b = shifts[(j+output_offset)%4][1]
        c = shifts[(j+output_offset)%4][2]
        iv_u = differences_pattern[j][0:16]
        iv_l = differences_pattern[j][16:32]
        ov_u = differences_pattern[j+1][0:16]
        ov_l = differences_pattern[j+1][16:32]

        active_bits = [(0, k) for k, bit in enumerate(iv_u) if bit] + \
                    [(1, k) for k, bit in enumerate(iv_l) if bit]

        ov_u_final = [0]*16
        ov_l_final = [0]*16

        for layer, bit_idx in active_bits:
            temp_iv_u = [0]*16
            temp_iv_l = [0]*16
            if layer == 0:
                temp_iv_u[bit_idx] = 1
            else:
                temp_iv_l[bit_idx] = 1
            temp_ov_u = [0]*16
            temp_ov_l = [0]*16
            for k in range(16):
                xor_u = temp_iv_u[(k+a)%16] ^ temp_iv_u[k] ^ temp_iv_l[(k+c)%16]
                temp_ov_u[k] = xor_u
                xor_l = temp_iv_l[(k+a)%16] ^ temp_iv_l[k] ^ temp_iv_u[(k+b)%16]
                temp_ov_l[k] = xor_l

            ov_u_final = [ov_u_final[k] | temp_ov_u[k] for k in range(16)]
            ov_l_final = [ov_l_final[k] | temp_ov_l[k] for k in range(16)]

        differences_pattern[j+1][0:16] = ov_u_final
        differences_pattern[j+1][16:32] = ov_l_final

    M_T = M_T_for_linear_layer(output_offset+1)
    mask_1 = np.zeros(32, dtype=int)
    for i in range(32):
        if differences_pattern[1][i]==1:
            mask_1|=M_T[i]

    mask_k2 = mask_1|differences_pattern[2]
    mask_k3 = np.zeros(32, dtype=int)
    M_T = M_T_for_linear_layer(output_offset+2)
    for i in range(32):
        if mask_k2[i]==1:
            mask_k3|=M_T[i]

    mask_k3|=differences_pattern[3]

    nb_keybits[0][sum(differences_pattern[1])]+=1
    nb_keybits[1][sum(differences_pattern[1])+np.sum(mask_k2)]+=1
    nb_keybits[2][sum(differences_pattern[1])+np.sum(mask_k2)+np.sum(mask_k3)]+=1

print("After one round")
print("#key bits | #times")
for k in range(32):
    if nb_keybits[0][k]!=0:
        print("   ",k,"     ",nb_keybits[k])

print("After two rounds")
print("#key bits | #times")
for k in range(2*32):
    if nb_keybits[1][k]!=0:
        print("   ",k,"     ",nb_keybits[k])

print("After three rounds")
print("#key bits | #times")
for k in range(3*32):
    if nb_keybits[2][k]!=0:
        print("   ",k,"     ",nb_keybits[k])